In [1]:


import pandas as pd
import os
from collections import defaultdict

raw_data_folder = 'raw_data_messy/work'
excel_files = [f for f in os.listdir(raw_data_folder) if f.endswith('.csv')]

rename_columns_dict = {
    "COMPANYNAME": ["COMPANY", "COMPANYNAMEFOREMAILS", "COMPANY.1"],
    "COMPANYADDRESS": ["ADDRESS", "ADDRESSES", "COMPANYSTREETADDRESS", "STREETADDRESS"],
    "COMPANYLOCATION": ["LOCATION", "LEADLOCATION"],
    "COMPANYCITY": ["CITY", "CITY.1"],
    "COMPANYSTATE": ["STATE", "STATE.1"],
    "COMPANYZIPCODE": ["ZIPCODE", "ZIPCODE", "COMPANYZIPCODE"],
    "COMPANYCOUNTRY": ["COUNTRY", "COUNTRY.1"],
    "COMPANYLINKEDIN": ["COMPANYLINKEDINURL", "LINKEDINURL", "LINKEDIN", "COMPANYLINKEDINPAGE"],
    "JOBTITLE": ["TITLE", "LEADTITLES", "JOBTITLE"],
    "LINKEDIN": ["PERSONLINKEDINURL"],
    "FACEBOOK": ["FACEBOOKURL", "COMPANYFACEBOOKPAGE"],
    "MOBILE_PHONE": ["PERSONALPHONE", "PERSONALPHONE1", "PERSONALPHONE2", "CELL", "MOBILEPHONE", "PHONENUMBER", "PHONE", "PERSONALPHONE.1"],
    "WORKPHONENUMBER": ["COMPANYPHONE", "CORPORATEPHONE", "OTHERPHONE", "COMPANYPHONENUMBERS", "WORKDIRECTPHONE", "WORKPHONE", "CORPORATEPHONE.1"],
    "WEBSITE": ["COMPANYWEBSITE", "COPYOFCOMPANYWEBSITE", "WEBSITE.1"],
    "INDUSTRY": ["COMPANYINDUSTRY", "COMPANYSECTOR"],
    "COMPANYREVENUE": ["ANNUALREVENUE"],
    "COMPANYREVENUERANGE": ["REVENUERANGE"],
    "EMPLOYEECOUNT": ["EMPLOYEES", "EMPLOYEESCOUNT", "COMPANYSIZE", "EMPLOY"],
    "COMPANYTWITTER": ["TWITTERURL", "COMPANYTWITTERPAGE"],
    "COMPANYFUNDING": ["TOTALFUNDING"],
    "EMAIL": ["EMAIL.1"]
}

rename_columns_dict_inversed = {
    alias: original
    for original, aliases in rename_columns_dict.items()
    for alias in aliases
}

def make_unique_with_counter(items):
    seen_counts = {}
    result = []

    items = [x.split('#')[0] for x in items]
    for item in items:
        if item not in seen_counts:
            # First time we see this item
            seen_counts[item] = 0
            result.append(item)
        else:
            # We've seen this item before, increment the count
            seen_counts[item] += 1
            new_item = f"{item}#{seen_counts[item]}"
            result.append(new_item)

    return result

def _clean_headers(headers: list):
    headers = [x.upper().replace(' ', '').replace("_", "").replace("A.", "").replace("B.", "") for x in headers]
    headers = make_unique_with_counter(headers)

    return headers

def _drop_columns(temp_df: pd.DataFrame) -> pd.DataFrame:
    columns_to_drop = ["SKILLS", "", "KEYWORDS", "COUNTRY", "TYPE", "SOURCESHEET"]
    temp_df = temp_df.drop(columns=columns_to_drop, errors='ignore')
    temp_df = temp_df.drop(columns=[col for col in temp_df.columns if "UNNAMED" in col])
    return temp_df

columns_dict = {}
df = pd.DataFrame()
for file in excel_files:
    file_path = os.path.join(raw_data_folder, file)
    temp_df = pd.read_csv(file_path, dtype="string")
    if file == "www.csv":
        www_columns = pd.read_csv('columns_to_keep/www_columns.csv')['column_names'].tolist()
        temp_df = temp_df[www_columns]
    if file == "Combined Master File 2_1.csv":
        masterfile_2_1_columns = pd.read_csv('columns_to_keep/masterfile_2_1_columns.csv')['column_names'].tolist()
        temp_df = temp_df[masterfile_2_1_columns]
    temp_df = temp_df.replace("-", None)
    temp_df.insert(0, "SOURCEFILE", file)
    temp_df = (temp_df.dropna(how="all").dropna(axis=1, how="all")) # drop empty rows and columns
    temp_df.columns = _clean_headers(temp_df.columns.tolist())
    temp_df = _drop_columns(temp_df) # drop unwanted columns
    temp_df = temp_df.rename(columns=rename_columns_dict_inversed)
    temp_df.columns = _clean_headers(temp_df.columns.tolist())
    temp_df = temp_df.rename(columns=rename_columns_dict_inversed)
    temp_df.columns = _clean_headers(temp_df.columns.tolist())
    for name_col in ["NAME", "OWNERNAME", "PERSONNAME", "FULLNAME"]:
        if name_col in temp_df.columns:
            name_splits = temp_df[name_col].str.split(' ', n=1, expand=True)
            temp_df['FIRSTNAME'] = name_splits[0].str.strip()
            temp_df['LASTNAME'] = name_splits[1].str.strip()
            temp_df = temp_df.drop(columns=[name_col])
    for column in temp_df.columns:
        columns_dict[column] = columns_dict.get(column, []) + [file]
    df = pd.concat([df, temp_df]).reset_index(drop=True)
df["INDUSTRY"] = df["INDUSTRY"].str.title()

groups = defaultdict(list)
for col in df.columns:
    if "#" in col and col.split("#")[-1].isdigit():
        base = "#".join(col.split("#")[:-1])
        groups[base].append(col)

for base, suffixes in groups.items():
    suffixes_sorted = sorted(suffixes, key=lambda x: int(x.split("#")[-1]))
    if base not in df.columns:
        df[base] = None
    for col in suffixes_sorted:
        df[base] = df[base].fillna(df[col])
df = df.drop(columns=[x for x in df.columns.tolist() if "#" in x])



In [2]:
df.columns.tolist()


['SOURCEFILE',
 'FIRSTNAME',
 'LASTNAME',
 'JOBTITLE',
 'EMAIL',
 'MOBILEPHONE',
 'COMPANYLINKEDIN',
 'FACEBOOK',
 'COMPANYTWITTER',
 'WORKPHONENUMBER',
 'INDUSTRY',
 'COMPANYNAME',
 'WEBSITE',
 'COMPANYADDRESS',
 'COMPANYZIPCODE',
 'EMPLOYEECOUNT',
 'COMPANYCITY',
 'COMPANYSTATE',
 'COMPANYCOUNTRY',
 'COMPANYREVENUE',
 'COMPANYLOCATION',
 'COMPANYFOUNDEDAT',
 'COMPANYREVENUERANGE',
 'COMPANYFUNDING',
 'LATESTFUNDINGSTAGE']

In [3]:
from datetime import datetime
datetime_now = datetime.now().strftime("%Y-%m-%d")
df.to_csv(f"csv_outputs/merge_messy_work_df_{datetime_now}.csv", index=False)


In [4]:

# # pre cleaning done to one of the files
# # clean raw_data_messy/DO NOT USE/Rob Volmer Project master File - 157k.csv
# import pandas as pd

# # Read the 8 tables from the CSV file, each starting at a different row
# data1 = pd.read_csv("raw_data_messy/DO NOT USE/Rob Volmer Project master File - 157k.csv", dtype="string", nrows=8028)
# data1["source"] = 1
# data2 = pd.read_csv("raw_data_messy/DO NOT USE/Rob Volmer Project master File - 157k.csv", dtype="string", skiprows=8029, nrows=18598-8030)
# data2["source"] = 2
# data3 = pd.read_csv("raw_data_messy/DO NOT USE/Rob Volmer Project master File - 157k.csv", dtype="string", skiprows=18598, nrows=37010-18599)
# data3["source"] = 3
# data4 = pd.read_csv("raw_data_messy/DO NOT USE/Rob Volmer Project master File - 157k.csv", dtype="string", skiprows=37010, nrows=43710-37011)
# data4["source"] = 4
# data5 = pd.read_csv("raw_data_messy/DO NOT USE/Rob Volmer Project master File - 157k.csv", dtype="string", skiprows=43710, nrows=50346-43711)
# data5["source"] = 5
# data9 = pd.read_csv("raw_data_messy/DO NOT USE/Rob Volmer Project master File - 157k.csv", dtype="string", skiprows=50346, nrows=64897-50347)
# data9["source"] = 9
# data6 = pd.read_csv("raw_data_messy/DO NOT USE/Rob Volmer Project master File - 157k.csv", dtype="string", skiprows=64897, nrows=85918-64898-1) # -1 for the random date 
# data6["source"] = 6
# data7 = pd.read_csv("raw_data_messy/DO NOT USE/Rob Volmer Project master File - 157k.csv", dtype="string", skiprows=85918, nrows=100985-85919)
# data7["source"] = 7
# data8 = pd.read_csv("raw_data_messy/DO NOT USE/Rob Volmer Project master File - 157k.csv", dtype="string", skiprows=100985)
# data8["source"] = 8

# data = pd.concat([data1, data2, data3, data4, data5, data9, data6, data7, data8, ], ignore_index=True)
# data = data.drop(columns=["source"])
# print(len(data))
# data.to_csv("raw_data_messy/work/Rob Volmer Project master File - 157k RX ADJUSTED.csv", index=False)

In [5]:


# import pandas as pd

# data = pd.read_csv("raw_data_messy/work/Rob Volmer Project master File - 157k RX ADJUSTED.csv", dtype="string")
# mask =(data.index >= 109975) & (data.index <= 115076)
# to_move = data[mask]
# leave_alone = data[~mask]


# # Shift the misaligned columns in to_move
# to_move = to_move.copy()

# # Store original values first
# original_city = to_move["City"].copy()
# original_company_address = to_move["Company Address"].copy()
# original_company_city = to_move["Company City"].copy()
# original_state = to_move["State"].copy()
# original_country = to_move["Country"].copy()

# # Move values according to the instructions
# to_move["Company Address"] = original_city
# to_move["Company Country"] = original_company_address
# to_move["Zip Code"] = original_company_city
# to_move["Company City"] = original_state
# to_move["Company State"] = original_country

# to_move

# # Merge the two dataframes back together
# merged_df = pd.concat([leave_alone, to_move], ignore_index=False)
# merged_df = merged_df.sort_index()

# # Export to CSV
# merged_df.to_csv("raw_data_messy/work/Rob Volmer Project master File - 157k RX ADJUSTED 2.csv", index=False)


In [6]:
# import pandas as pd

# data = pd.read_csv("raw_data_messy/DO NOT USE/allin1match file no 2 - Sheet1.csv", dtype="string")

# mask =(data.index >= 81320) & (data.index <= 82725)
# to_move = data[mask]
# leave_alone = data[~mask]

# to_move = to_move.copy()

# # Store original values first
# original_city = to_move["Company City"].copy()
# original_address = to_move["Company Address"].copy()
# original_state = to_move["Company State"].copy()
# original_country = to_move["Company Country"].copy()

# # Move values according to the instructions
# to_move["Company Address"] = original_city
# to_move["Company Country"] = original_address
# to_move["Company City"] = original_state
# to_move["Company State"] = original_country

# # Merge the two dataframes back together
# data = pd.concat([leave_alone, to_move], ignore_index=False)
# data = data.sort_index()

# mask =(data.index >= 81418) & (data.index <= 82314)
# to_move = data[mask]
# leave_alone = data[~mask]

# to_move = to_move.copy()

# # Store original values first
# original_city = to_move["Company City"].copy()
# original_address = to_move["Company Address"].copy()
# original_state = to_move["Company State"].copy()
# original_country = to_move["Company Country"].copy()

# # Move values according to the instructions
# to_move["Company Address"] = original_country
# to_move["Company Country"] = original_state
# to_move["Company City"] = original_address
# to_move["Company State"] = original_city

# # Merge the two dataframes back together
# data = pd.concat([leave_alone, to_move], ignore_index=False)
# data = data.sort_index()

# # Export to CSV
# data.to_csv("raw_data_messy/work/allin1match file no 2 - Sheet1 FIXED.csv", index=False)
